# Chapter 4 &mdash; Formal Acceptance: $\delta$, $\hat{\delta}$, `accepts`

**Concept 10 of the Chapter 4 decomposition:** *Formal Acceptance: $\delta$, $\hat{\delta}$, and the `accepts` Predicate*

One function per layer: step a symbol, run a string, run from any state, decide.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter4/Concept-Delta-Hat-Acceptance/Concept-Delta-Hat-Acceptance.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
# Run this cell first. It works both on Colab and on your own machine.
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
import sys

try:                       # -- are we on Colab? --
    import google.colab
    OWN_INSTALL = False
except ImportError:
    OWN_INSTALL = True

if OWN_INSTALL:
    # Running from Jove/Chapter<N>/Concept-<Name>/ : reach the Jove root.
    sys.path[0:0] = ['../..', '../../3rdparty',
                     '../../..', '../../../3rdparty',
                     '..', '../3rdparty', '.']
else:
    ! if [ ! -d Jove ]; then git clone -q https://github.com/ganeshutah/Jove Jove; fi
    sys.path.append('./Jove')
    sys.path.append('./Jove/jove')

# -- imports needed by this notebook --
from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
#~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~~
print("Jove loaded. Ready.")

## 1. The idea


Acceptance is defined by three functions, and Jove has one per layer:

* **$\delta$ = `step_dfa(D,q,c)`** &mdash; one state, one symbol, one next state;
* **$\hat{\delta}$ = `run_dfa_h(D,s,q)`** &mdash; a *string* from an *arbitrary* state,
  defined recursively with basis $\hat{\delta}(D,q,\varepsilon)=q$;
* **`run_dfa(D,s)`** &mdash; $\hat{\delta}$ from $q_0$;
* **`accepts_dfa(D,s)`** &mdash; run, then test membership in $F$.

The `_h` suffix means *helper* and marks the general-starting-state version.

## 2. Definitions

### The machine

In [ ]:
D = md2mc('''DFA
I : 0 -> I
I : 1 -> F
F : 0 -> F
F : 1 -> I
''')

### $\hat{\delta}$, written out

The recursion is on the **string**, not the machine.

In [ ]:
def delta_hat(D, q, s):
    return q if s == '' else delta_hat(D, step_dfa(D, q, s[0]), s[1:])

## 3. Tests

The basis case $\hat{\delta}(D,q,\varepsilon)=q$ &mdash; which is why an initial-and-final state accepts $\varepsilon$.

In [ ]:
for q in sorted(D["Q"]):
    print("delta_hat(D, %s, '') = %s" % (q, delta_hat(D, q, '')))
assert all(delta_hat(D, q, '') == q for q in D["Q"])

Our recursion agrees with Jove's `run_dfa` and `run_dfa_h`.

In [ ]:
for s in ['', '1', '10', '111', '1010']:
    mine, jove = delta_hat(D, D["q0"], s), run_dfa(D, s)
    print("s=%-7r delta_hat=%-3s run_dfa=%-3s agree=%s" % (s, mine, jove, mine == jove))
assert all(delta_hat(D, D["q0"], s) == run_dfa(D, s)
           for s in ['', '1', '10', '111', '1010', '0000'])

`run_dfa_h` starts anywhere &mdash; useful for reasoning mid-computation.

In [ ]:
print("from F on '1'  ->", run_dfa_h(D, '1', 'F'))
print("from I on '11' ->", run_dfa_h(D, '11', 'I'))
assert run_dfa_h(D, '', 'F') == 'F'

And acceptance is just membership in $F$.

In [ ]:
for s in ['1', '11', '101']:
    print("%-6r ends in %-3s in F? %-5s accepts_dfa=%s"
          % (s, run_dfa(D, s), run_dfa(D, s) in D["F"], accepts_dfa(D, s)))
assert all((run_dfa(D, s) in D["F"]) == accepts_dfa(D, s) for s in ['1','11','101',''])

## 4. Exercises


1. Rewrite `delta_hat` to recurse on the *last* symbol instead of the first.
   Does it still agree?
2. What is $\hat{\delta}(D,q,s)$ when $s$ has 1000 symbols? Try it &mdash; and explain
   the error.
3. Why does the definition need `run_dfa_h` at all, when `run_dfa` exists?

In [ ]:
# Your work for the exercises above.